# 🎯 [DAT06_M05] 정산 결과의 회계적 일관성 검증 및 신뢰도 검증 엔진 개발
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT06/dat06_m05_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT06_M05 |
| **미션 주제** | 정산 결과의 회계적 일관성 검증 및 신뢰도 검증 엔진 개발 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
미국 국방재무회계국(DFAS)은 반복적인 재무·정산 업무에 업무 자동화(RPA)를 도입해 사람이 하던 취합과 점검을 줄이는 사업을 추진해 왔다. 이 공개 사례의 학습 초점은 반복 업무 분해, 입력과 트리거, 자동 처리 순서, 오류 복구, 실행 로그, 시간 절감이다. 본 프로젝트는 이를 바탕으로 교육용 가상 부대 24개의 월간 결산 자료(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)를 코드로 만들고, 파일 대신 메모리 안의 가상 문서와 표로 모의한다. 학습자는 업무 분해, 다형식 파싱, 검증 룰 엔진, 이상 원인 추천, 일관성 검증, 승인·반려 대시보드, 3단계 사람 결재선, 대량 배치 시험, AAR 보정, 최종 통합을 차례로 수행한다. 최종 목표는 월간 정산·보고·취합을 한 번에 실행하되 발간은 반드시 사람이 결재하도록 하는 원클릭 자동화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> 💡 **[산출물(Deliverables) 디렉토리 경로 및 실습 안내]**
> - **아키텍처 관점의 표준 산출물:** 과제 명세에 표기된 src/connectors/..., docs/..., dist/... 등의 경로는 본 솔루션이 실무 배포 환경에서 갖추게 되는 **소프트웨어 표준 아키텍처 구조**를 의미합니다.
> - **노트북 중심 실습:** 별도의 폴더나 외부 파이썬 파일(.py)을 수작업으로 생성하실 필요가 없습니다. 본 노트북의 각 미션별 실습 셀을 AI 코파일럿과 함께 완성하고 검증하면 해당 산출물 요건이 자동으로 충족됩니다.
> - **미션 10 최종 통합 패키징:** 각 미션 셀에서 구현된 단위 모듈과 명세는 **[미션 10]** 실행 시 단일 소프트웨어 패키지(dist/dat06_package/)와 운영 매뉴얼(docs/dat06_final_manual.md)로 일괄 결합되어 자동 추출됩니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 정산 결과의 회계적 일관성 검증 및 신뢰도 검증 엔진 개발
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

정산 결과의 회계적 일관성 검증 및 신뢰도 검증 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **정산 결과의 회계적 일관성을 검증하고 부대별 신뢰도 점수를 산출하는 엔진을 입력-분석-검토-결과 4개 화면으로 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **재고 산식, 전월 기말과 당월 기초의 연속성, 부대 합계와 분류 합계의 교차 대조, 장비 가동률 범위를 검사하고 오류 8건을 주입해 검출 여부 확인하고 유효성을 점검하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ui/dat06_app.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 실무자 친화적 디자인 및 조작 편의성

#### 미션 PASS 조건

- 정량 기준: 핵심 화면 4종(입력-분석-검토-결과) 구현 완료


- 진행 기준: UI 프로토타입 로컬 서버 구동 및 전체 페이지 네비게이션 확인


- 검증 방법: 사용성 평가(UX Walkthrough) 및 인터랙션 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""결산 결과의 회계적 일관성(재고 산식·월 연속성·합계 교차 대조·가동률 범위)을 검증해 부대별 신뢰도 점수를 산출하고, 입력-분석-검토-결과 4개 화면을 텍스트 화면으로 구현한다."""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 월간 결산 원장 생성기 (실제 부대명·품목·수량·재무 정보 없음)
# 가상 대대 24개 x (물자 불출 / 수리부속 소모 / 탄약 정산) 품목 원장과 장비 가동 현황을 만든다.
# 재고 산식: 기말 = 기초 + 수령 + 반납 - 불출,  다음 달 기초 = 이번 달 기말.
# ---------------------------------------------------------------------------
UNITS = [f"가상{i:02d}대대" for i in range(1, 25)]
ITEMS = {"물자불출": ["급식류", "피복류", "유류", "정비소모품"],
         "수리부속": ["필터류", "베어링류", "전장품", "타이어류"],
         "탄약정산": ["훈련탄A", "훈련탄B", "모의탄"]}
UNIT_CODE = {"급식류": "BX", "피복류": "EA", "유류": "L", "정비소모품": "EA", "필터류": "EA", "베어링류": "EA",
             "전장품": "EA", "타이어류": "EA", "훈련탄A": "RD", "훈련탄B": "RD", "모의탄": "RD"}
LEDGER_COLS = ["month", "unit", "cat", "item", "open", "recv", "issue", "ret", "close"]


def make_ledger(n_months=2, seed=60):
    """n_months개월 원장(list of dict). 둘째 달 기초는 첫째 달 기말과 같다."""
    rng = np.random.default_rng(seed)
    rows, prev = [], {}
    for m in range(n_months):
        for u in UNITS:
            for cat, items in ITEMS.items():
                for it in items:
                    base = int(rng.integers(80, 400))
                    op = prev.get((u, it), base * 3)
                    recv = int(rng.integers(0, base))
                    issue = int(rng.integers(base // 4, base // 2 + 1))
                    ret = int(issue * rng.uniform(0, 0.15))
                    cl = op + recv + ret - issue
                    rows.append({"month": f"2026-{m + 7:02d}", "unit": u, "cat": cat, "item": it,
                                 "open": op, "recv": recv, "issue": issue, "ret": ret, "close": cl})
                    prev[(u, it)] = cl
    return rows


def make_equip(n_months=2, seed=61):
    rng = np.random.default_rng(seed)
    out = []
    for m in range(n_months):
        for u in UNITS:
            hold = int(rng.integers(20, 60))
            out.append({"month": f"2026-{m + 7:02d}", "unit": u, "hold": hold, "operable": int(hold * rng.uniform(0.78, 0.97))})
    return out

# ---------- 화면 1: 입력 ----------
led = pd.DataFrame(make_ledger(2))
eq = pd.DataFrame(make_equip(2))
cur = led[led["month"] == "2026-08"].copy()
prev = led[led["month"] == "2026-07"].set_index(["unit", "item"])
rng = np.random.default_rng(5)
faults = []                                        # 가상 오류 주입 8건
for i in rng.choice(cur.index, 8, replace=False):
    kind = ["산식", "연속성", "산식", "연속성"][len(faults) % 4]
    if kind == "산식":
        cur.loc[i, "close"] += int(rng.integers(5, 40))
    else:
        cur.loc[i, "open"] += int(rng.integers(5, 40)); cur.loc[i, "close"] = cur.loc[i, "open"] + cur.loc[i, "recv"] + cur.loc[i, "ret"] - cur.loc[i, "issue"]
    faults.append((cur.loc[i, "unit"], cur.loc[i, "item"], kind))
eq2 = eq[eq["month"] == "2026-08"].copy()
eq2.iloc[3, eq2.columns.get_loc("operable")] = eq2.iloc[3]["hold"] + 4   # 가동 대수가 보유 대수 초과
SCREEN1 = f"원장 {len(cur)}행(월 2026-08), 장비 현황 {len(eq2)}행, 전월 원장 {len(prev)}행 적재 / 양식 검사 통과"
print("[화면 1. 입력]", SCREEN1)

# ---------- 화면 2: 분석(일관성 검사 4종) ----------
issues = []
for r in cur.itertuples():
    if r.open + r.recv + r.ret - r.issue != r.close:
        issues.append((r.unit, r.item, "C1 재고 산식 불일치", abs(r.open + r.recv + r.ret - r.issue - r.close)))
    if prev.loc[(r.unit, r.item), "close"] != r.open:
        issues.append((r.unit, r.item, "C2 전월 기말 != 당월 기초", abs(prev.loc[(r.unit, r.item), "close"] - r.open)))
unit_sum = cur.groupby("unit")[["open", "recv", "ret", "issue", "close"]].sum()
cat_sum = cur.groupby(["unit", "cat"])[["open", "recv", "ret", "issue", "close"]].sum().groupby("unit").sum()
for u in unit_sum.index:                           # C3 교차 대조: 부대 합계 = 분류 합계의 합
    if not (unit_sum.loc[u] == cat_sum.loc[u]).all():
        issues.append((u, "-", "C3 합계 교차 대조 불일치", 0))
for r in eq2.itertuples():
    if not (0 <= r.operable <= r.hold):
        issues.append((r.unit, "장비", "C4 가동 대수가 보유 대수 초과", r.operable - r.hold))
iss = pd.DataFrame(issues, columns=["부대", "품목", "검사", "차이"])
print(f"[화면 2. 분석] 검사 4종 수행, 불일치 {len(iss)}건")
print(iss["검사"].value_counts().to_string())

# ---------- 신뢰도 점수 ----------
score = pd.Series(100.0, index=UNITS)
W = {"C1": 12, "C2": 10, "C3": 15, "C4": 8}
for r in iss.itertuples():
    score[r.부대] -= W[r.검사[:2]]
score = score.clip(lower=0)
grade = pd.cut(score, [-1, 69, 89, 100], labels=["C(재제출)", "B(검토)", "A(양호)"])

# ---------- 화면 3: 검토 ----------
print(f"\n[화면 3. 검토] 신뢰도 100점 미만 부대와 검토 대기 사항 [교육용가상자료]")
low = iss.groupby("부대").agg(건수=("검사", "count"), 유형=("검사", lambda s: ", ".join(sorted(set(x[:2] for x in s))))).join(score.rename("점수"))
print(low.sort_values("점수").to_string())
print("검토자는 불일치 건별로 '수정 요청 / 정상 확인' 중 선택하며, 수정은 제출 부대가 한다(시스템이 값을 임의로 바꾸지 않는다).")

# ---------- 화면 4: 결과 ----------
detected = {(u, i) for u, i, k in faults}
caught = {(r.부대, r.품목) for r in iss.itertuples() if r.검사[:2] in ("C1", "C2")}
print(f"\n[화면 4. 결과] 부대 {len(score)}개 중 A {int((grade == 'A(양호)').sum())} / B {int((grade == 'B(검토)').sum())} / C {int((grade == 'C(재제출)').sum())}")
print(f"평균 신뢰도 {score.mean():.1f}점, 주입 오류 {len(faults)}건 중 검출 {len(detected & caught)}건({len(detected & caught) / len(faults):.0%}), 가동률 범위 오류 {int((iss['검사'].str[:2] == 'C4').sum())}건")
ok = len(detected & caught) == len(faults)
print(f"\n목표(핵심 화면 4종 구현, 일관성 검증 엔진 동작): {'달성' if ok else '미달'} (화면 4종: 입력/분석/검토/결과)")


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
[화면 1. 입력] 원장 264행(월 2026-08), 장비 현황 24행, 전월 원장 264행 적재 / 양식 검사 통과
[화면 2. 분석] 검사 4종 수행, 불일치 9건
검사
C2 전월 기말 != 당월 기초     4
C1 재고 산식 불일치          4
C4 가동 대수가 보유 대수 초과    1

[화면 3. 검토] 신뢰도 100점 미만 부대와 검토 대기 사항 [교육용가상자료]
        건수      유형    점수
부대                      
가상16대대   2  C1, C2  78.0
가상12대대   1      C1  88.0
가상13대대   1      C1  88.0
가상20대대   1      C1  88.0
가상01대대   1      C2  90.0
가상07대대   1      C2  90.0
가상19대대   1      C2  90.0
가상04대대   1      C4  92.0
검토자는 불일치 건별로 '수정 요청 / 정상 확인' 중 선택하며, 수정은 제출 부대가 한다(시스템이 값을 임의로 바꾸지 않는다).

[화면 4. 결과] 부대 24개 중 A 20 / B 4 / C 0
평균 신뢰도 96.0점, 주입 오류 8건 중 검출 8건(100%), 가동률 범위 오류 1건

목표(핵심 화면 4종 구현, 일관성 검증 엔진 동작): 달성 (화면 4종: 입력/분석/검토/결과)
```
</details>
